<a href="https://colab.research.google.com/github/akhilabusalih/flyrank-ml-internship/blob/copilot%2Fcomplete-week5-ml08-assignment/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/akhilabusalih/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb)

This notebook reproduces the Week 5 ML-08 model comparison using the real anonymized dataset and the existing project pipeline, without synthetic replacements or hard-coded metrics.

## 1. Method Choice and Why

The target is binary classification: `is_declining_label = (trend_direction == "down")`.

This maps to a yes/no prediction task: given historical content signals, estimate whether a page belongs to the declining class.

Three model families are compared because they provide a useful progression:
- **Logistic Regression**: linear, interpretable baseline classifier.
- **Decision Tree**: nonlinear model with readable split logic.
- **Random Forest**: ensemble model that can capture more complex interactions.

No model is assumed superior in advance. Selection is made only after measured evaluation on the held-out test set.

In [6]:
from __future__ import annotations

import importlib.util
import json
import subprocess
import sys
import os
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

# If running in Google Colab and the repository is not yet cloned, clone it
if 'colab' in sys.modules or os.path.exists('/content'):
    repo_name = "flyrank-ml-internship"
    if not Path(repo_name).exists():
        print(f"Cloning repository {repo_name}...")
        subprocess.run(["git", "clone", "https://github.com/akhilabusalih/flyrank-ml-internship.git"], check=True)

    # Change directory to the repository path containing scripts and data
    repo_dir = Path.cwd() / repo_name / "work"
    if repo_dir.exists():
        os.chdir(str(repo_dir))

def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "scripts").exists() and (candidate / "data").exists():
            return candidate
    raise RuntimeError("Could not locate repository root containing scripts/ and data/")

ROOT = find_repo_root(Path.cwd())
RAW_PATH = ROOT / "data" / "raw" / "content_refresh_anonymized.csv"
FEATURE_PATH = ROOT / "data" / "processed" / "refresh_feature_vector.csv"
BASELINE_PATH = ROOT / "data" / "processed" / "baseline_refresh_queue.csv"
PREDICTIONS_PATH = ROOT / "data" / "processed" / "model_predictions.csv"
RESULTS_PATH = ROOT / "outputs" / "model_results.json"

SCRIPT_DIR = ROOT / "scripts"
if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))

assert RAW_PATH.exists(), f"Real dataset missing at {RAW_PATH}"
print(f"Repository root: {ROOT}")
print(f"Raw dataset: {RAW_PATH}")
print(f"Python: {sys.version.split()[0]}")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")

Cloning repository flyrank-ml-internship...
Repository root: /content/flyrank-ml-internship/work/flyrank-ml-internship
Raw dataset: /content/flyrank-ml-internship/work/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv
Python: 3.13.15
pandas: 2.2.3
numpy: 2.1.3


## 2. Data and Split Design

This workflow uses the prepared feature vector produced from the real dataset.

Split design is **client-aware holdout** (not plain random row split):
- training set and test set are separated by `client_id` when feasible,
- this reduces optimistic leakage from client-specific patterns appearing in both train and test,
- fallback is stratified row holdout only when client-level split cannot satisfy class constraints.

Target variable:
- `is_declining_label`

Model feature policy:
- allowed features come from `MODEL_NUMERIC_FEATURES` and `MODEL_CATEGORICAL_FEATURES`,
- leakage/identifier columns (`trend_direction`, `trend_pct`, `content_id`, `client_id`) are excluded from model features by assertion.

In [7]:
# Load project utilities and training module without editing existing scripts
import ml_utils

train_spec = importlib.util.spec_from_file_location("train_model", ROOT / "scripts" / "03_train_model.py")
train_model = importlib.util.module_from_spec(train_spec)
assert train_spec and train_spec.loader
train_spec.loader.exec_module(train_model)

if FEATURE_PATH.exists():
    feature_frame = pd.read_csv(FEATURE_PATH)
    print(f"Loaded existing feature vector: {FEATURE_PATH} ({len(feature_frame):,} rows)")
else:
    print(f"Prepared feature vector not present yet: {FEATURE_PATH}")

Prepared feature vector not present yet: /content/flyrank-ml-internship/work/flyrank-ml-internship/data/processed/refresh_feature_vector.csv


## 3. Baseline vs Models

Run the existing pipeline on the real dataset:
`01_prepare_features.py` → `02_baseline_score.py` → `03_train_model.py`

Then read measured metrics from generated outputs (no manual metric entry).

In [8]:
def run_script(script_name: str) -> None:
    script_path = ROOT / "scripts" / script_name
    cmd = [sys.executable, str(script_path)]
    print(f"Running: {' '.join(cmd)}")
    subprocess.run(cmd, check=True, cwd=str(ROOT))


run_script("01_prepare_features.py")
run_script("02_baseline_score.py")
run_script("03_train_model.py")

for path in [FEATURE_PATH, BASELINE_PATH, PREDICTIONS_PATH, RESULTS_PATH]:
    assert path.exists(), f"Expected artifact missing: {path}"
    print(f"OK: {path}")

Running: /usr/bin/python3 /content/flyrank-ml-internship/work/flyrank-ml-internship/scripts/01_prepare_features.py
Running: /usr/bin/python3 /content/flyrank-ml-internship/work/flyrank-ml-internship/scripts/02_baseline_score.py
Running: /usr/bin/python3 /content/flyrank-ml-internship/work/flyrank-ml-internship/scripts/03_train_model.py
OK: /content/flyrank-ml-internship/work/flyrank-ml-internship/data/processed/refresh_feature_vector.csv
OK: /content/flyrank-ml-internship/work/flyrank-ml-internship/data/processed/baseline_refresh_queue.csv
OK: /content/flyrank-ml-internship/work/flyrank-ml-internship/data/processed/model_predictions.csv
OK: /content/flyrank-ml-internship/work/flyrank-ml-internship/outputs/model_results.json


In [17]:
feature_frame = pd.read_csv(FEATURE_PATH)
baseline_frame = pd.read_csv(BASELINE_PATH)
prediction_frame = pd.read_csv(PREDICTIONS_PATH)
with open(RESULTS_PATH, "r", encoding="utf-8") as f:
    model_results = json.load(f)

forbidden = {"trend_direction", "trend_pct", "content_id", "client_id"}
model_feature_list = ml_utils.MODEL_NUMERIC_FEATURES + ml_utils.MODEL_CATEGORICAL_FEATURES
for forbidden_feature in forbidden:
    assert forbidden_feature not in model_feature_list, f"Leakage feature detected: {forbidden_feature}"

X, feature_columns = train_model.build_feature_matrix(feature_frame)
y = feature_frame["is_declining_label"].astype(int)
train_idx, test_idx, split_strategy = train_model.make_client_aware_split(feature_frame, y)

train_clients = set(feature_frame.iloc[train_idx]["client_id"].astype(str))
test_clients = set(feature_frame.iloc[test_idx]["client_id"].astype(str))
client_overlap = train_clients.intersection(test_clients)

print(f"Rows: {len(feature_frame):,}")
print(f"Target: is_declining_label (positive rate={y.mean():.3f})")
print(f"Train rows: {len(train_idx):,}")
print(f"Test rows: {len(test_idx):,}")
print(f"Split strategy: {split_strategy}")
print(f"Model feature count (after encoding): {len(feature_columns)}")
print(f"Client overlap between train/test: {len(client_overlap)}")
if split_strategy == "client_holdout":
    assert len(client_overlap) == 0, "Client holdout expected zero overlap"

baseline_metrics = model_results["baseline"]
rows = [
    {
        "model": "baseline",
        "accuracy": baseline_metrics.get("baseline_accuracy"),
        "precision": baseline_metrics.get("baseline_precision"),
        "recall": baseline_metrics.get("baseline_recall"),
        "f1": baseline_metrics.get("baseline_f1"),
        "precision@20": baseline_metrics.get("baseline_precision_at_20"),
        "precision@50": baseline_metrics.get("baseline_precision_at_50"),
        "precision@100": baseline_metrics.get("baseline_precision_at_100"),
        "roc_auc": baseline_metrics.get("baseline_roc_auc"),
        "average_precision": baseline_metrics.get("baseline_average_precision"),
    }
]

for model_name, metrics in model_results["models"].items():
    rows.append({
        "model": model_name,
        "accuracy": metrics.get("accuracy"),
        "precision": metrics.get("precision"),
        "recall": metrics.get("recall"),
        "f1": metrics.get("f1"),
        "precision@20": metrics.get("precision_at_20"),
        "precision@50": metrics.get("precision_at_50"),
        "precision@100": metrics.get("precision_at_100"),
        "roc_auc": metrics.get("roc_auc"),
        "average_precision": metrics.get("average_precision"),
    })

comparison_df = pd.DataFrame(rows)
comparison_df = comparison_df[[
    "model", "accuracy", "precision", "recall", "f1",
    "precision@20", "precision@50", "precision@100", "roc_auc", "average_precision"
]]
comparison_df

Rows: 30,000
Target: is_declining_label (positive rate=0.542)
Train rows: 27,675
Test rows: 2,325
Split strategy: client_holdout
Model feature count (after encoding): 52
Client overlap between train/test: 0


,model,accuracy,precision,recall,f1,precision@20,precision@50,precision@100,roc_auc,average_precision
0,baseline,0.608602,0.498551,0.189219,0.274322,0.15,0.24,0.36,0.626892,0.467607
1,decision_tree,0.676559,0.568559,0.716172,0.633885,0.45,0.58,0.62,0.741520,0.575319
2,logistic_regression,0.660645,0.565934,0.566557,0.566245,0.35,0.40,0.44,0.700291,0.521542
3,random_forest,0.672258,0.560996,0.743674,0.639546,0.65,0.74,0.72,0.750030,0.618219


## 4. Model Selection

Primary selection criterion is **Precision@50**, because the workflow prioritizes a limited top queue of items for manual refresh review.

The selected model here is the one with the highest measured Precision@50 on this held-out test set. This is a dataset-and-split-specific result, not a universal model ranking.

In [18]:
model_only = comparison_df[comparison_df["model"] != "baseline"].copy()
selected_row = model_only.sort_values(["precision@50", "average_precision", "roc_auc"], ascending=False).iloc[0]
selected_model = selected_row["model"]

print(f"Model with highest measured Precision@50 on this test split: {selected_model}")
print()
print("All measured metrics for context:")
comparison_df.sort_values("precision@50", ascending=False).reset_index(drop=True)

Model with highest measured Precision@50 on this test split: random_forest

All measured metrics for context:


,model,accuracy,precision,recall,f1,precision@20,precision@50,precision@100,roc_auc,average_precision
0,random_forest,0.672258,0.560996,0.743674,0.639546,0.65,0.74,0.72,0.750030,0.618219
1,decision_tree,0.676559,0.568559,0.716172,0.633885,0.45,0.58,0.62,0.741520,0.575319
2,logistic_regression,0.660645,0.565934,0.566557,0.566245,0.35,0.40,0.44,0.700291,0.521542
3,baseline,0.608602,0.498551,0.189219,0.274322,0.15,0.24,0.36,0.626892,0.467607


## 5. Error Analysis

Analyze real false positives and false negatives on the test set using the selected model's predicted probabilities.

Patterns are described only when supported by measured differences in this holdout.

In [19]:
test_frame = feature_frame.iloc[test_idx].copy().reset_index(drop=True)
test_preds = prediction_frame[prediction_frame["split"] == "test"].copy().reset_index(drop=True)

test_eval = test_frame.merge(
    test_preds[["content_id", f"prob_{selected_model}", "best_model_probability"]],
    on="content_id",
    how="inner",
)
assert len(test_eval) > 0, "No matched test rows for error analysis"

selected_prob_col = f"prob_{selected_model}"
test_eval["predicted_label"] = (test_eval[selected_prob_col] >= 0.5).astype(int)
test_eval["actual_label"] = test_eval["is_declining_label"].astype(int)

cm = confusion_matrix(test_eval["actual_label"], test_eval["predicted_label"])
cm_df = pd.DataFrame(cm, index=["actual_0", "actual_1"], columns=["pred_0", "pred_1"])
print("Confusion matrix (selected model, test set):")
cm_df

Confusion matrix (selected model, test set):


,pred_0,pred_1
actual_0,1180,236
actual_1,296,613


In [20]:
def error_type(row):
    if row["actual_label"] == 1 and row["predicted_label"] == 1:
        return "true_positive"
    if row["actual_label"] == 0 and row["predicted_label"] == 0:
        return "true_negative"
    if row["actual_label"] == 0 and row["predicted_label"] == 1:
        return "false_positive"
    return "false_negative"


test_eval["error_type"] = test_eval.apply(error_type, axis=1)

false_positives = test_eval[test_eval["error_type"] == "false_positive"].copy()
false_negatives = test_eval[test_eval["error_type"] == "false_negative"].copy()

print(f"False positives: {len(false_positives):,}")
print(f"False negatives: {len(false_negatives):,}")

cols_preview = [
    "content_id", "client_id", "actual_label", "predicted_label", selected_prob_col,
    "impressions_90d", "sessions_90d", "content_age_days", "days_since_last_update",
    "avg_position", "ctr", "engagement_rate", "content_type", "competition_level"
]

print("Sample false positives:")
display(false_positives[cols_preview].head(5))
print("Sample false negatives:")
display(false_negatives[cols_preview].head(5))

False positives: 236
False negatives: 296
Sample false positives:


,content_id,client_id,actual_label,predicted_label,prob_random_forest,impressions_90d,sessions_90d,content_age_days,days_since_last_update,avg_position,ctr,engagement_rate,content_type,competition_level
4,content_95d488a56079,client_f74efabef1,0,1,0.608286,564,15,140,8,12.1,0.89,0.0,keyword article,LOW
13,content_bec800684271,client_f74efabef1,0,1,0.606859,56,1,91,20,27.4,1.79,0.0,keyword article,LOW
15,content_d7cbd76b788d,client_f74efabef1,0,1,0.625555,17992,36,144,20,6.4,0.11,0.0,keyword article,LOW
60,content_c9c5cbaa8586,client_1a6562590e,0,1,0.606434,342,2,347,104,19.7,0.00,0.0,keyword article,LOW
80,content_77fad6753ab4,client_f74efabef1,0,1,0.633501,1773,13,123,20,7.8,0.28,0.0,keyword article,LOW


Sample false negatives:


,content_id,client_id,actual_label,predicted_label,prob_random_forest,impressions_90d,sessions_90d,content_age_days,days_since_last_update,avg_position,ctr,engagement_rate,content_type,competition_level
1,content_326fa2fa449f,client_98a3ab7c34,1,0,0.460304,4,2,91,1,8.3,0.00,0.0,keyword article,LOW
7,content_48724397d104,client_d4735e3a26,1,0,0.449224,38,2,305,211,2.5,5.26,0.0,keyword article,unknown
23,content_421272323961,client_d4735e3a26,1,0,0.366540,6,2,116,8,9.3,16.67,0.0,feedly article,unknown
27,content_d2e655334ee2,client_d4735e3a26,1,0,0.448271,8,1,489,20,2.6,12.50,0.0,keyword article,LOW
34,content_4148b1c5a86e,client_0b918943df,1,0,0.288239,7,3,314,20,11.3,0.00,0.0,feedly article,unknown


In [21]:
numeric_checks = [
    "impressions_90d", "sessions_90d", "content_age_days", "days_since_last_update",
    "avg_position", "ctr", "engagement_rate", "scroll_rate", "ai_traffic_pct"
]

summary_numeric = (
    test_eval.groupby("error_type")[numeric_checks]
    .median()
    .T
    .round(3)
)
summary_numeric

error_type,false_negative,false_positive,true_negative,true_positive
impressions_90d,8.00,1014.000,4.000,524.00
sessions_90d,1.00,9.000,1.000,4.00
content_age_days,293.00,117.000,290.000,117.00
days_since_last_update,20.00,20.000,20.000,20.00
avg_position,5.50,10.450,4.300,11.10
ctr,0.00,0.190,0.000,0.06
engagement_rate,0.00,0.000,0.000,0.00
scroll_rate,33.33,2.595,16.475,0.00
ai_traffic_pct,0.00,0.000,0.000,0.00


In [22]:
summary_content_type = (
    test_eval.groupby(["error_type", "content_type"]).size()
    .rename("count")
    .reset_index()
    .sort_values(["error_type", "count"], ascending=[True, False])
)
summary_competition = (
    test_eval.groupby(["error_type", "competition_level"]).size()
    .rename("count")
    .reset_index()
    .sort_values(["error_type", "count"], ascending=[True, False])
)

print("Error type by content_type:")
display(summary_content_type.head(20))
print("Error type by competition_level:")
display(summary_competition.head(20))

Error type by content_type:


,error_type,content_type,count
0,false_negative,feedly article,183
1,false_negative,keyword article,113
3,false_positive,keyword article,230
2,false_positive,feedly article,6
4,true_negative,feedly article,759
5,true_negative,keyword article,421
7,true_positive,keyword article,603
6,true_positive,feedly article,10


Error type by competition_level:


,error_type,competition_level,count
3,false_negative,unknown,201
1,false_negative,LOW,86
0,false_negative,HIGH,5
2,false_negative,MEDIUM,4
5,false_positive,LOW,182
4,false_positive,HIGH,29
6,false_positive,MEDIUM,17
7,false_positive,unknown,8
11,true_negative,unknown,823
9,true_negative,LOW,262


In [23]:
def rel_diff(a: float, b: float) -> float | None:
    # Use a safety threshold to avoid division by near-zero values
    if abs(b) < 0.01:
        return None
    return (a - b) / b

insights = []

if {"false_negative", "true_positive"}.issubset(set(summary_numeric.columns)):
    fn_tp = summary_numeric[["false_negative", "true_positive"]]
    for col in ["impressions_90d", "sessions_90d", "content_age_days", "days_since_last_update", "avg_position", "ctr", "engagement_rate"]:
        val_fn = float(fn_tp.loc[col, "false_negative"])
        val_tp = float(fn_tp.loc[col, "true_positive"])
        delta = rel_diff(val_fn, val_tp)
        if delta is not None and abs(delta) >= 0.2:
            direction = "higher" if delta > 0 else "lower"
            insights.append(f"For {col}, false negatives show {direction} median values than true positives (relative difference {delta:+.1%}).")

if {"false_positive", "true_negative"}.issubset(set(summary_numeric.columns)):
    fp_tn = summary_numeric[["false_positive", "true_negative"]]
    for col in ["impressions_90d", "sessions_90d", "content_age_days", "days_since_last_update", "avg_position", "ctr", "engagement_rate"]:
        val_fp = float(fp_tn.loc[col, "false_positive"])
        val_tn = float(fp_tn.loc[col, "true_negative"])
        delta = rel_diff(val_fp, val_tn)
        if delta is not None and abs(delta) >= 0.2:
            direction = "higher" if delta > 0 else "lower"
            insights.append(f"For {col}, false positives show {direction} median values than true negatives (relative difference {delta:+.1%}).")

if insights:
    print("Observed error-pattern notes (associational, not causal):")
    for line in insights:
        print(f"- {line}")
else:
    print("No strong numeric separation (>20% median relative difference with safe denominators) was established across the checked error groups in this test split.")
    print("This means we should avoid making stronger pattern claims from this run.")

Observed error-pattern notes (associational, not causal):
- For impressions_90d, false negatives show lower median values than true positives (relative difference -98.5%).
- For sessions_90d, false negatives show lower median values than true positives (relative difference -75.0%).
- For content_age_days, false negatives show higher median values than true positives (relative difference +150.4%).
- For avg_position, false negatives show lower median values than true positives (relative difference -50.5%).
- For ctr, false negatives show lower median values than true positives (relative difference -100.0%).
- For impressions_90d, false positives show higher median values than true negatives (relative difference +25250.0%).
- For sessions_90d, false positives show higher median values than true negatives (relative difference +800.0%).
- For content_age_days, false positives show lower median values than true negatives (relative difference -59.7%).
- For avg_position, false positives show

## 6. Feature Importance

Interpretations below are **associations with model predictions**, not causal effects.

We inspect:
- absolute coefficients for Logistic Regression,
- impurity-based feature importance for Decision Tree and Random Forest,
- and check for suspicious leakage-like top features.

In [25]:
full_X, full_feature_columns = train_model.build_feature_matrix(feature_frame)
full_y = feature_frame["is_declining_label"].astype(int)
models = train_model.build_models()

# Fit models using only the training split
train_X_subset = full_X.iloc[train_idx]
train_y_subset = full_y.iloc[train_idx]

for name, model in models.items():
    model.fit(train_X_subset, train_y_subset)

log_model = models["logistic_regression"].named_steps["model"]
coef_df = pd.DataFrame({
    "feature": full_feature_columns,
    "coefficient": log_model.coef_[0],
    "abs_coefficient": np.abs(log_model.coef_[0]),
}).sort_values("abs_coefficient", ascending=False)

print("Top Logistic Regression coefficients (by absolute magnitude):")
display(coef_df.head(15))

dt_importance = pd.DataFrame({
    "feature": full_feature_columns,
    "importance": models["decision_tree"].feature_importances_,
}).sort_values("importance", ascending=False)

rf_importance = pd.DataFrame({
    "feature": full_feature_columns,
    "importance": models["random_forest"].feature_importances_,
}).sort_values("importance", ascending=False)

print("Top Decision Tree importances:")
display(dt_importance.head(15))
print("Top Random Forest importances:")
display(rf_importance.head(15))

Top Logistic Regression coefficients (by absolute magnitude):


,feature,coefficient,abs_coefficient
5,log_impressions_90d,1.658755,1.658755
3,word_count,1.607656,1.607656
4,char_count,-1.351979,1.351979
6,log_clicks_90d,-0.656016,0.656016
14,avg_position,-0.404586,0.404586
9,days_with_impressions,-0.251329,0.251329
11,content_age_days,-0.248888,0.248888
45,impression_tier_low,0.245641,0.245641
7,log_sessions_90d,-0.243358,0.243358
51,position_tier_top_3,-0.190942,0.190942


Top Decision Tree importances:


,feature,importance
9,days_with_impressions,0.432751
11,content_age_days,0.249213
10,days_with_sessions,0.103021
14,avg_position,0.100849
16,scroll_rate,0.038793
4,char_count,0.026155
13,ctr,0.023785
6,log_clicks_90d,0.023342
12,days_since_last_update,0.002081
26,main_intent_informational,0.000010


Top Random Forest importances:


,feature,importance
9,days_with_impressions,0.134951
5,log_impressions_90d,0.129377
14,avg_position,0.109203
11,content_age_days,0.092048
4,char_count,0.038676
32,age_tier_365+,0.036847
6,log_clicks_90d,0.036572
3,word_count,0.035406
13,ctr,0.035156
16,scroll_rate,0.033876


In [26]:
suspicious_tokens = ["trend_direction", "trend_pct", "content_id", "client_id"]

def suspicious_top_features(df: pd.DataFrame, top_n: int = 25):
    top_features = df.head(top_n)["feature"].astype(str).tolist()
    flagged = [f for f in top_features if any(token in f for token in suspicious_tokens)]
    return flagged

flagged_lr = suspicious_top_features(coef_df)
flagged_dt = suspicious_top_features(dt_importance)
flagged_rf = suspicious_top_features(rf_importance)

print("Leakage-style top-feature check:")
print(f"- Logistic top features flagged: {flagged_lr}")
print(f"- Decision Tree top features flagged: {flagged_dt}")
print(f"- Random Forest top features flagged: {flagged_rf}")
assert not flagged_lr and not flagged_dt and not flagged_rf, "Suspicious top features detected"

Leakage-style top-feature check:
- Logistic top features flagged: []
- Decision Tree top features flagged: []
- Random Forest top features flagged: []


## 7. Limitations

- This is observational modeling, so associations should not be interpreted as causal effects.
- Client distribution can vary, and holdout performance depends on which clients are in train vs test.
- Results reflect this historical snapshot and may shift with future data updates (distribution shift risk).
- Performance is conditional on the current preparation logic and feature definitions.
- Precision@K is operationally useful but sensitive to the chosen K (e.g., @20 vs @50 vs @100).
- Rankings and thresholded predictions should support editorial review, not replace it.

## 8. Reproducibility Checklist

- [x] Fixed random seed (`RANDOM_STATE = 42` in training pipeline)
- [x] Real dataset used: `data/raw/content_refresh_anonymized.csv`
- [x] Client-aware split preserved where feasible
- [x] Leakage/ID features excluded from model-feature definitions
- [x] Baseline and ML models evaluated on the same held-out test split
- [x] Metrics loaded from actual generated outputs (no hard-coded results)
- [x] Generated processed/output artifacts are gitignored and not committed
- [x] Notebook designed to run top-to-bottom

In [27]:
assert RAW_PATH.exists()
assert FEATURE_PATH.exists()
assert BASELINE_PATH.exists()
assert PREDICTIONS_PATH.exists()
assert RESULTS_PATH.exists()

required_metric_cols = [
    "accuracy", "precision", "recall", "f1", "precision@20", "precision@50",
    "precision@100", "roc_auc", "average_precision"
]
for col in required_metric_cols:
    assert col in comparison_df.columns, f"Missing metric column: {col}"

print("Reproducibility guards passed.")
comparison_df.sort_values("precision@50", ascending=False).reset_index(drop=True)

Reproducibility guards passed.


,model,accuracy,precision,recall,f1,precision@20,precision@50,precision@100,roc_auc,average_precision
0,random_forest,0.672258,0.560996,0.743674,0.639546,0.65,0.74,0.72,0.750030,0.618219
1,decision_tree,0.676559,0.568559,0.716172,0.633885,0.45,0.58,0.62,0.741520,0.575319
2,logistic_regression,0.660645,0.565934,0.566557,0.566245,0.35,0.40,0.44,0.700291,0.521542
3,baseline,0.608602,0.498551,0.189219,0.274322,0.15,0.24,0.36,0.626892,0.467607
